In [12]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama
from langchain_classic.chains import RetrievalQA
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

print("1. Cargando el documento...")
loader = TextLoader("../Data/Preprocesado/Informacion_Rag/base_conocimiento_salud_mental_rag.txt", encoding="utf-8")
documento = loader.load()

# Trocear el documento en fragmentos
print("2. Fragmentando el texto...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
fragmentos = text_splitter.split_documents(documento)

1. Cargando el documento...
2. Fragmentando el texto...


In [14]:
# Definimos la ruta donde se guardará la base de datos
PERSIST_DIRECTORY = "../models/IA/chroma_db"

print("3. Configurando embeddings y base de datos vectorial...")
modelo_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Comprobamos si la base de datos vectorial ya existe en disco
if not os.path.exists(PERSIST_DIRECTORY):
    print("Creando y guardando base de datos vectorial en disco por primera vez...")
    vectorstore = Chroma.from_documents(
        documents=fragmentos, 
        embedding=modelo_embeddings,
        persist_directory=PERSIST_DIRECTORY  # <-- Guarda la BD en esta carpeta
    )
else:
    print("Cargando base de datos vectorial persistida existente...")
    vectorstore = Chroma(
        persist_directory=PERSIST_DIRECTORY, # <-- Carga la BD desde la carpeta
        embedding_function=modelo_embeddings
    )

# Convertimos la base de datos en un retriever
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
compressor = CrossEncoderReranker(model=model, top_n=3)

# 4. Combinar retrieval y reranking en un único retriever
retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=base_retriever
)

print("4. Conectando el LLM local con Ollama...")

# 1. Inicializar el modelo local de Ollama
llm = ChatOllama(
    model="llama3.1",
    temperature=0
)

# 2. Unir el LLM local con nuestro recuperador vectorial
cadena_rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

3. Configurando embeddings y base de datos vectorial...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9810.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Creando y guardando base de datos vectorial en disco por primera vez...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10164.88it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


4. Conectando el LLM local con Ollama...


In [10]:
print("\n--- SISTEMA RAG LISTO ---")
pregunta = "¿Que es la depresion?"

respuesta = cadena_rag.invoke(pregunta)
print(f"\nPregunta: {pregunta}")
print(f"Respuesta del RAG: {respuesta['result']}")


--- SISTEMA RAG LISTO ---

Pregunta: ¿Que es la depresion?
Respuesta del RAG: La depresión es un trastorno mental común que implica un estado de ánimo deprimido o pérdida de interés o placer durante periodos prolongados, y puede afectar a relaciones, estudios, trabajo y otras áreas de la vida.


In [11]:
pregunta = "Que es la ansiedad?"
respuesta = cadena_rag.invoke(pregunta)
print(f"\nPregunta: {pregunta}")
print(f"Respuesta del RAG: {respuesta['result']}")


Pregunta: Que es la ansiedad?
Respuesta del RAG: La ansiedad es un estado emocional caracterizado por un sentimiento de miedo, preocupación o inquietud excesiva que puede ser difícil de controlar. Puede manifestarse de diferentes maneras y niveles de intensidad, y puede interferir con la vida cotidiana de las personas.
